In [2]:

!apt-get update -qq
!apt-get install -y tesseract-ocr tesseract-ocr-por -qq


!pip install pytesseract -q

print(" Instalação concluída com sucesso!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package tesseract-ocr-por.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-por_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-por (1:4.1.0-2) ...
Setting up tesseract-ocr-por (1:4.1.0-2) ...
 Instalação concluída com sucesso!


In [3]:
import cv2
import pytesseract


versao = pytesseract.get_tesseract_version()
print(f"Tesseract pronto para uso! Versão: {versao}")

Tesseract pronto para uso! Versão: 5.3.4


In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')

meu_drive = '/content/drive/MyDrive'
pastas = os.listdir(meu_drive)

print("\n Pastas encontradas no seu Meu Drive:")
for p in pastas:
    if 'Comprovantes' in p or 'gasto' in p:
        print(f" Nome exato: '{p}'")

KeyboardInterrupt: 

In [ ]:
import os

subpasta = 'Uber'

caminho_pasta = f'/content/drive/MyDrive/Comprovantes de gastos - Caso Marcia Fidelis/{subpasta}'

if os.path.exists(caminho_pasta):
    ficheiros = os.listdir(caminho_pasta)
    imagens = [f for f in ficheiros if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Encontradas {len(imagens)} imagens na pasta '{subpasta}':")
    print(imagens[:5])
else:
    print(f" Pasta não encontrada em: {caminho_pasta}")

In [ ]:
!apt-get install -y poppler-utils -qq
!pip install pdf2image -q

import os
from pdf2image import convert_from_path
from matplotlib import pyplot as plt

subpasta = 'Uber'
caminho_pasta = f'/content/drive/MyDrive/Comprovantes de gastos - Caso Marcia Fidelis/{subpasta}'

ficheiros = os.listdir(caminho_pasta)
pdfs = [f for f in ficheiros if f.lower().endswith('.pdf')]

print(f"Encontrados {len(pdfs)} arquivos PDF na pasta '{subpasta}'")

if len(pdfs) > 0:
    caminho_pdf = os.path.join(caminho_pasta, pdfs[0])
    paginas = convert_from_path(caminho_pdf)

    imagem_pdf = paginas[0]

    plt.imshow(imagem_pdf)
    plt.axis('off')
    plt.show()

    print("Primeiro PDF convertido e exibido:", pdfs[0])

In [ ]:
import pytesseract

texto_extraido = pytesseract.image_to_string(imagem_pdf, lang='por')

print("--- TEXTO EXTRAÍDO DO COMPROVANTE ---")
print(texto_extraido)

In [ ]:
import re

# Busca por padrões de valor em dinheiro (ex: R$ 25,90 ou 25,90)
padrao_valor = r'R?\$\s*(\d+[\.,]\d{2})'
valores = re.findall(padrao_valor, texto_extraido)

# Busca por padrões de data (ex: 12/05/2024 ou 12-05-2024)
padrao_data = r'(\d{2}[\/\.-]\d{2}[\/\.-]\d{4})'
datas = re.findall(padrao_data, texto_extraido)

print("Valores encontrados:", valores)
print("Datas encontradas:", datas)


In [ ]:
import re

# Padrão expandido para datas (ex: 12/05/2024, 12/05/24, 12 de maio de 2024)
padrao_data_amplo = r'(\d{1,2}\s+(?:de\s+)?[A-Za-zçÇéÉãÃâÂ\.]+\s+(?:de\s+)?\d{2,4}|\d{2}[\/\.-]\d{2}[\/\.-]\d{2,4})'

datas = re.findall(padrao_data_amplo, texto_extraido, re.IGNORECASE)

print("Novas datas encontradas:", datas)

As imagens estao nao extensão pdf, o pytesseract
ficou lento. usar o pypdf


In [ ]:
!pip install pypdf -q

import os
import re
import pandas as pd
from pypdf import PdfReader

subpasta = 'Uber'
caminho_pasta = f'/content/drive/MyDrive/Comprovantes de gastos - Caso Marcia Fidelis/{subpasta}'

padrao_valor = r'R?\$\s*(\d+[\.,]\d{2})'
padrao_data = r'(\d{1,2}\s+(?:de\s+)?[A-Za-zçÇéÉãÃâÂ\.]+\s+(?:de\s+)?\d{2,4}|\d{2}[\/\.-]\d{2}[\/\.-]\d{2,4})'

dados = []

ficheiros = os.listdir(caminho_pasta)
pdfs = [f for f in ficheiros if f.lower().endswith('.pdf')]

for pdf in pdfs:
    caminho_pdf = os.path.join(caminho_pasta, pdf)
    reader = PdfReader(caminho_pdf)

    texto = ""
    for page in reader.pages:
        texto += page.extract_text() or ""

    valores = re.findall(padrao_valor, texto)
    datas = re.findall(padrao_data, texto, re.IGNORECASE)

    valor_final = valores[0] if valores else 'Não encontrado'
    data_final = datas[0] if datas else 'Não encontrada'

    dados.append({
        'Arquivo': pdf,
        'Data': data_final,
        'Valor': valor_final
    })

df = pd.DataFrame(dados)
print(f"Processamento concluído para {len(df)} arquivos!")
df.head()

In [ ]:
from google.colab import files

files.download('/content/relatorio_comprovantes_uber.xlsx')

In [ ]:
import pandas as pd
from google.colab import files

df['Duplicado'] = df.duplicated(subset=['Data', 'Valor'], keep=False)

df_duplicados = df[df['Duplicado'] == True].sort_values(by=['Data', 'Valor'])

print(f"Total de registros com mesma data e valor encontrados: {len(df_duplicados)}")

caminho_duplicados = '/content/comprovantes_duplicados_uber.xlsx'
df.to_excel(caminho_duplicados, index=False)

files.download(caminho_duplicados)

df_duplicados.head(10)

In [ ]:
from google.colab import auth
import gspread
from google.auth import default

# Autenticação nativa e interativa do Google Colab
auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

nome_planilha = "Comprovantes ADV"
spreadsheet = gc.open(nome_planilha)

print(f"Planilha '{nome_planilha}' aberta com sucesso!")

for worksheet in spreadsheet.worksheets():
    dados = worksheet.get_all_values()
    if not dados or len(dados) <= 1:
        continue

    total_linhas = len(dados)
    total_colunas = len(dados[0])

    linha_soma = total_linhas + 1
    coluna_valor_letra = chr(64 + total_colunas)

    worksheet.update_cell(linha_soma, 1, "TOTAL")
    worksheet.update_cell(linha_soma, total_colunas, f"=SUM({coluna_valor_letra}2:{coluna_valor_letra}{total_linhas})")

    worksheet.format(f"A1:{coluna_valor_letra}1", {
        "textFormat": {"bold": True},
        "backgroundColor": {"red": 0.12, "green": 0.31, "blue": 0.47}
    })

    worksheet.format(f"A{linha_soma}:{coluna_valor_letra}{linha_soma}", {
        "textFormat": {"bold": True}
    })

print("Auto Soma adicionada e formatação concluída com sucesso em todas as abas!")

In [ ]:
# ============================================================
# PADRONIZAÇÃO E SOMA — COMPROVANTES ADV
# Google Colab + Google Sheets + gspread
# ============================================================

# ------------------------------------------------------------
# 1. INSTALAÇÃO / IMPORTAÇÕES
# ------------------------------------------------------------

!pip -q install --upgrade gspread google-auth

import re
import unicodedata
import gspread

from google.colab import auth
from google.auth import default
from gspread.utils import rowcol_to_a1


# ------------------------------------------------------------
# 2. AUTENTICAÇÃO
# ------------------------------------------------------------

print("🔐 Autenticando no Google...")

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

print("✅ Autenticação concluída.")


# ------------------------------------------------------------
# 3. ABRIR A PLANILHA
# ------------------------------------------------------------

NOME_PLANILHA = "Comprovantes ADV"

print(f"\n📂 Abrindo planilha: {NOME_PLANILHA}")

try:
    planilha = gc.open(NOME_PLANILHA)
    print("✅ Planilha aberta com sucesso.")

except Exception as e:
    print(f"❌ Erro ao abrir a planilha: {e}")
    raise


# ------------------------------------------------------------
# 4. FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def normalizar_texto(texto):
    """
    Remove acentos, espaços extras e coloca em minúsculas.
    Facilita encontrar cabeçalhos como:
    Valor, VALOR, valor, Valor (R$), etc.
    """

    if texto is None:
        return ""

    texto = str(texto).strip().lower()

    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )

    texto = re.sub(r"\s+", " ", texto)

    return texto


def encontrar_coluna(cabecalhos, nomes_procurados):
    """
    Procura dinamicamente uma coluna pelo nome do cabeçalho.
    """

    nomes_normalizados = [
        normalizar_texto(nome)
        for nome in nomes_procurados
    ]

    for indice, cabecalho in enumerate(cabecalhos, start=1):

        cabecalho_norm = normalizar_texto(cabecalho)

        # Correspondência exata
        if cabecalho_norm in nomes_normalizados:
            return indice

    # Segunda tentativa:
    # permite cabeçalhos como "Valor (R$)"
    for indice, cabecalho in enumerate(cabecalhos, start=1):

        cabecalho_norm = normalizar_texto(cabecalho)

        for nome in nomes_normalizados:

            if nome in cabecalho_norm:
                return indice

    return None


def converter_para_numero(valor):
    """
    Converte valores que podem estar como:

    150
    150.00
    150,00
    R$ 150,00
    R$ 1.250,50
    1.250,50

    para número Python.
    """

    if valor is None:
        return None

    valor = str(valor).strip()

    if valor == "":
        return None

    # Remove R$, espaços e outros caracteres
    valor = valor.replace("R$", "")
    valor = valor.replace(" ", "")

    # Caso brasileiro:
    # 1.250,50 -> 1250.50
    if "," in valor and "." in valor:
        valor = valor.replace(".", "")
        valor = valor.replace(",", ".")

    # Caso apenas vírgula:
    # 1250,50 -> 1250.50
    elif "," in valor:
        valor = valor.replace(",", ".")

    # Remove caracteres que não sejam número, ponto ou sinal
    valor = re.sub(r"[^0-9.\-]", "", valor)

    if valor in ("", "-", "."):
        return None

    try:
        return float(valor)

    except ValueError:
        return None


def descobrir_ultima_linha_de_dados(valores, coluna_valor, coluna_data):
    """
    Descobre a última linha realmente preenchida com dados,
    considerando as colunas de Data e Valor.

    Isso evita colocar o TOTAL no meio de uma tabela.
    """

    ultima = 1

    for numero_linha, linha in enumerate(valores, start=1):

        if numero_linha == 1:
            continue

        data = (
            linha[coluna_data - 1]
            if len(linha) >= coluna_data
            else ""
        )

        valor = (
            linha[coluna_valor - 1]
            if len(linha) >= coluna_valor
            else ""
        )

        if str(data).strip() != "" or str(valor).strip() != "":
            ultima = numero_linha

    return ultima


# ------------------------------------------------------------
# 5. FORMATAÇÃO
# ------------------------------------------------------------

def aplicar_formatacao(
    aba,
    ultima_linha_total,
    coluna_data,
    coluna_valor,
    numero_colunas
):

    print("   🎨 Aplicando formatação...")

    # ----------------------------------------
    # Fonte Arial em toda a área utilizada
    # ----------------------------------------

    aba.format(
        f"A1:{rowcol_to_a1(ultima_linha_total, numero_colunas)}",
        {
            "textFormat": {
                "fontFamily": "Arial",
                "fontSize": 10
            }
        }
    )

    # ----------------------------------------
    # CABEÇALHO
    # ----------------------------------------

    aba.format(
        f"A1:{rowcol_to_a1(1, numero_colunas)}",
        {
            "backgroundColor": {
                "red": 31 / 255,
                "green": 78 / 255,
                "blue": 121 / 255
            },
            "textFormat": {
                "fontFamily": "Arial",
                "fontSize": 10,
                "bold": True,
                "foregroundColor": {
                    "red": 1,
                    "green": 1,
                    "blue": 1
                }
            },
            "horizontalAlignment": "CENTER",
            "verticalAlignment": "MIDDLE"
        }
    )

    # ----------------------------------------
    # COLUNA DATA — CENTRALIZADA
    # ----------------------------------------

    if coluna_data:

        aba.format(
            f"{rowcol_to_a1(2, coluna_data)}:"
            f"{rowcol_to_a1(ultima_linha_total, coluna_data)}",
            {
                "horizontalAlignment": "CENTER",
                "verticalAlignment": "MIDDLE"
            }
        )

    # ----------------------------------------
    # COLUNA VALOR — DIREITA + MOEDA
    # ----------------------------------------

    if coluna_valor:

        aba.format(
            f"{rowcol_to_a1(2, coluna_valor)}:"
            f"{rowcol_to_a1(ultima_linha_total, coluna_valor)}",
            {
                "horizontalAlignment": "RIGHT",
                "verticalAlignment": "MIDDLE",
                "numberFormat": {
                    "type": "NUMBER",
                    "pattern": 'R$ #,##0.00'
                }
            }
        )

    # ----------------------------------------
    # LINHA TOTAL
    # ----------------------------------------

    aba.format(
        f"A{ultima_linha_total}:"
        f"{rowcol_to_a1(ultima_linha_total, numero_colunas)}",
        {
            "textFormat": {
                "fontFamily": "Arial",
                "fontSize": 10,
                "bold": True
            },
            "borders": {
                "bottom": {
                    "style": "DOUBLE",
                    "color": {
                        "red": 0,
                        "green": 0,
                        "blue": 0
                    }
                }
            }
        }
    )


# ------------------------------------------------------------
# 6. PROCESSAR CADA ABA
# ------------------------------------------------------------

abas = planilha.worksheets()

print(f"\n📑 Abas encontradas: {len(abas)}")

for aba in abas:

    print("\n" + "=" * 70)
    print(f"📋 PROCESSANDO ABA: {aba.title}")
    print("=" * 70)

    try:

        # ----------------------------------------------------
        # LER DADOS
        # ----------------------------------------------------

        valores = aba.get_all_values()

        if not valores:

            print("⚠️ Aba vazia. Pulando.")
            continue

        cabecalhos = valores[0]

        print(f"   Cabeçalhos encontrados: {cabecalhos}")

        numero_colunas = len(cabecalhos)

        # ----------------------------------------------------
        # IDENTIFICAR DATA
        # ----------------------------------------------------

        coluna_data = encontrar_coluna(
            cabecalhos,
            [
                "data",
                "data do pagamento",
                "data pagamento"
            ]
        )

        # ----------------------------------------------------
        # IDENTIFICAR VALOR
        # ----------------------------------------------------

        coluna_valor = encontrar_coluna(
            cabecalhos,
            [
                "valor",
                "valor pago",
                "valor do pagamento",
                "valor (r$)",
                "valor r$"
            ]
        )

        print(
            f"   📅 Coluna Data: "
            f"{coluna_data if coluna_data else 'NÃO ENCONTRADA'}"
        )

        print(
            f"   💰 Coluna Valor: "
            f"{coluna_valor if coluna_valor else 'NÃO ENCONTRADA'}"
        )

        # ----------------------------------------------------
        # VERIFICAÇÃO DE SEGURANÇA
        # ----------------------------------------------------

        if coluna_valor is None:

            print(
                "❌ Não foi possível identificar a coluna de Valor."
            )

            print(
                "   Nenhuma alteração de soma será feita nesta aba."
            )

            continue

        # ----------------------------------------------------
        # DESCOBRIR ÚLTIMA LINHA DE DADOS
        # ----------------------------------------------------

        ultima_linha_dados = descobrir_ultima_linha_de_dados(
            valores,
            coluna_valor,
            coluna_data if coluna_data else coluna_valor
        )

        print(
            f"   📌 Última linha de dados: "
            f"{ultima_linha_dados}"
        )

        # ----------------------------------------------------
        # NORMALIZAR VALORES DA COLUNA VALOR
        # ----------------------------------------------------

        print("   🔢 Convertendo valores para números...")

        valores_numericos = []

        for linha in range(2, ultima_linha_dados + 1):

            valor_original = (
                valores[linha - 1][coluna_valor - 1]
                if len(valores[linha - 1]) >= coluna_valor
                else ""
            )

            numero = converter_para_numero(valor_original)

            valores_numericos.append([numero])

            print(
                f"      Linha {linha}: "
                f"{valor_original!r} → {numero}"
            )

        # ----------------------------------------------------
        # ESCREVER VALORES NUMÉRICOS
        # ----------------------------------------------------

        if valores_numericos:

            intervalo_valores = (
                f"{rowcol_to_a1(2, coluna_valor)}:"
                f"{rowcol_to_a1(ultima_linha_dados, coluna_valor)}"
            )

            aba.update(
                intervalo_valores,
                valores_numericos,
                value_input_option="USER_ENTERED"
            )

        # ----------------------------------------------------
        # LINHA DO TOTAL
        # ----------------------------------------------------

        linha_total = ultima_linha_dados + 1

        celula_valor_total = rowcol_to_a1(
            linha_total,
            coluna_valor
        )

        # ----------------------------------------------------
        # TEXTO TOTAL NA PRIMEIRA COLUNA
        # ----------------------------------------------------

        aba.update(
            f"A{linha_total}",
            [["TOTAL"]],
            value_input_option="USER_ENTERED"
        )

        # ----------------------------------------------------
        # FÓRMULA DE SOMA
        # ----------------------------------------------------

        primeira_celula_dados = rowcol_to_a1(
            2,
            coluna_valor
        )

        ultima_celula_dados = rowcol_to_a1(
            ultima_linha_dados,
            coluna_valor
        )

        formula = (
            f"=SUM("
            f"{primeira_celula_dados}:"
            f"{ultima_celula_dados}"
            f")"
        )

        print(
            f"   🧮 Fórmula do total: "
            f"{celula_valor_total} = {formula}"
        )

        aba.update(
            celula_valor_total,
            [[formula]],
            value_input_option="USER_ENTERED"
        )

        # ----------------------------------------------------
        # FORMATAÇÃO
        # ----------------------------------------------------

        aplicar_formatacao(
            aba,
            linha_total,
            coluna_data,
            coluna_valor,
            numero_colunas
        )

        print(
            f"   ✅ Aba '{aba.title}' concluída."
        )

    except Exception as e:

        print(
            f"❌ ERRO na aba '{aba.title}': {e}"
        )


# ------------------------------------------------------------
# 7. FINALIZAÇÃO
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎉 PROCESSAMENTO FINALIZADO")
print("=" * 70)

print(
    "\nVerifique as três abas no Google Sheets para confirmar "
    "a posição do TOTAL, a soma e a formatação."
)

In [ ]:
import os

# 1. Criar e acessar uma pasta exclusiva para o projeto (fora do Google Drive)
%cd /content
!mkdir -p projeto_recibos
%cd /content/projeto_recibos

# 2. Copiar o relatório gerado para dentro da pasta do projeto
!cp /content/comprovantes_duplicados_uber.xlsx /content/projeto_recibos/ 2>/dev/null || true

# 3. Configurar usuario no Git
!git config --global user.email "paulamlleal@gmail.com"
!git config --global user.name "Paula Albuquerque"

# 4. Inicializar o repositório limpo nesta pasta
!git init
!git branch -M main

# 5. Adicionar os arquivos locais do projeto
!git add .

# 6. Realizar o commit profissional com a mensagem escolhida
!git commit -m "feat: leitura e estruturação de dados via OCR/pypdf para escritório jurídico"

In [ ]:
!git remote add origin https://SEU_TOKEN_AQUI@github.com/seu_usuario/automacao-recibos-juridico.git
!git push -u origin main

In [ ]:
import os

# 1. Mudar para a pasta isolada do projeto
%cd /content
!mkdir -p projeto_recibos
%cd /content/projeto_recibos

# 2. Copiar o arquivo gerado para a pasta do projeto
!cp /content/comprovantes_duplicados_uber.xlsx /content/projeto_recibos/ 2>/dev/null || true

# 3. Configurar assinatura dos commits
!git config --global user.email "paulamlleal@gmail.com"
!git config --global user.name "Paula Albuquerque"

# 4. Inicializar e garantir branch main
!git init
!git branch -M main

# 5. Adicionar arquivos e criar commit técnico de ETL
!git add .
!git commit -m "feat(etl): pipeline de extracao, transformacao e carga de comprovantes para escritorio juridico" || true

In [5]:
import os

# 1. Criar a pasta isolada do projeto e acessar
!mkdir -p /content/projeto_recibos
%cd /content/projeto_recibos

# 2. Copiar o seu arquivo de código Python ou Notebook (.ipynb/.py) do Colab para a pasta do projeto
!cp /content/*.ipynb /content/projeto_recibos/ 2>/dev/null || true
!cp /content/*.py /content/projeto_recibos/ 2>/dev/null || true

# Copiar também a planilha de saída (caso ainda esteja na pasta temporária)
!cp /content/comprovantes_duplicados_uber.xlsx /content/projeto_recibos/ 2>/dev/null || true

# 3. Configurar assinatura e inicializar Git
!git config --global user.email "paulamlleal@gmail.com"
!git config --global user.name "Paula Albuquerque"
!git init
!git branch -M main

# 4. Adicionar todos os arquivos encontrados e registrar o commit
!git add .
!git commit -m "feat(etl): adiciona codigo do pipeline de extracao e carga de comprovantes" || true

# 5. Conectar com o GitHub da conta PaullaAllbuquerque
# Substitua 'SEU_TOKEN_AQUI' pelo seu Personal Access Token ativo (ghp_...)
TOKEN = "SEU_TOKEN_AQUI"
USERNAME = "PaullaAllbuquerque"
REPO_NAME = "etl-comprovantes-juridico"

remote_url = f"https://{TOKEN}@github.com/{USERNAME}/{REPO_NAME}.git"
os.system(f"git remote set-url origin {remote_url} 2>/dev/null || git remote add origin {remote_url}")

# 6. Enviar tudo para o GitHub
!git push -u origin main --force

/content/projeto_recibos
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/projeto_recibos/.git/
On branch main

Initial commit

nothing to commit (create/copy files and use "git add" to track)
error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/PaullaAllbuquerque/etl-comprovantes-juridico.git'
